In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import json
import logging
from collections import Counter
from pathlib import Path

# When run from notebooks/, hop up to the repo root so `from data...` and
# `from symbols...` both resolve.
if Path.cwd().name == "notebooks":
    os.chdir("..")
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

print("cwd:", Path.cwd())

import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt

Image.MAX_IMAGE_PIXELS = None
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)

# Repo modules.
from data.coco_loader import load_split
from symbols.call_symbol_localizer import (
    IsolatedSymbolClient,
    DEFAULT_PROJECT_ID,
    DEFAULT_LOCATION,
    DEFAULT_LOCALIZATION_ENDPOINT_ID,
    DEFAULT_CLASSIFICATION_ENDPOINT_ID,
    DEFAULT_GCS_BUCKET,
    DEFAULT_CREDENTIALS_FILE,
    _resolve_credentials_file,
)

## 1. Configure credentials + endpoints

The client needs three things:

1. **Service account JSON** for GCP auth. By default at `credentials/inference-428300-7af7f5da75dc.json`, or pointed at by `GOOGLE_APPLICATION_CREDENTIALS`.
2. **Two Vertex AI endpoint resource paths** (`projects/.../endpoints/<id>`), pre-set in `symbols/call_symbol_localizer.py`.
3. **A GCS bucket** to upload local image files to so the endpoint can fetch them (`public-bobyard`).

In [ ]:
credentials_file = _resolve_credentials_file(explicit="")
assert Path(credentials_file).is_file(), credentials_file

print("credentials_file:        ", credentials_file)
print("project_id:              ", DEFAULT_PROJECT_ID)
print("location:                ", DEFAULT_LOCATION)
print("localization endpoint:   ", DEFAULT_LOCALIZATION_ENDPOINT_ID)
print("classification endpoint: ", DEFAULT_CLASSIFICATION_ENDPOINT_ID)
print("gcs bucket:              ", DEFAULT_GCS_BUCKET)

## 2. Instantiate the client

One client is reused for every call in this notebook. At the very end we call `client.cleanup_uploads()` to delete the temporary GCS blobs that were uploaded during this session.

In [ ]:
NMS_IOU_THRESH = 0.5
CONF_THRESH    = 0.0

client = IsolatedSymbolClient(
    credentials_file=credentials_file,
    project_id=DEFAULT_PROJECT_ID,
    location=DEFAULT_LOCATION,
    localization_endpoint_id=DEFAULT_LOCALIZATION_ENDPOINT_ID,
    classification_endpoint_id=DEFAULT_CLASSIFICATION_ENDPOINT_ID,
    gcs_bucket=DEFAULT_GCS_BUCKET,
    nms_iou_thresh=NMS_IOU_THRESH,
    conf_thresh=CONF_THRESH,
)
print("client ready")

## 3. Pick a test image

Default: first image in the `test` split. Change `SPLIT` / `IMG_INDEX` to inspect another.

In [ ]:
SPLIT     = "test"
IMG_INDEX = 0

dataset_root = (Path.cwd() / "../datasets/keypoints.v56-lat_only_baseline.coco").resolve()
images, _ = load_split(dataset_root / SPLIT)
img_id = sorted(images.keys())[IMG_INDEX]
record = images[img_id]

print(f"file_name:  {record.file_name}")
print(f"resolution: {record.width} x {record.height}")
print(f"local path: {record.path}")

## 4. Run localization (with on-disk cache)

The first run uploads the image to GCS, calls the localization endpoint, and caches the raw response under `results/symbols/<split>_id<id>/`. Subsequent runs read from the cache.

Set `FORCE_RECOMPUTE = True` to bypass the cache.

In [ ]:
FORCE_RECOMPUTE = False

CACHE_DIR = Path(f"results/symbols/{SPLIT}_id{img_id:03d}")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
raw_path = CACHE_DIR / "raw_localizations.json"

if raw_path.exists() and not FORCE_RECOMPUTE:
    raw_localizations = json.loads(raw_path.read_text())
    print(f"loaded {len(raw_localizations)} detections from cache: {raw_path}")
else:
    print("calling localization endpoint...")
    raw_localizations = client.localize(str(record.path))
    raw_path.write_text(json.dumps(raw_localizations, indent=2))
    print(f"saved {len(raw_localizations)} detections to: {raw_path}")

## 5. Inspect the raw API response

Look at the structure of a single record so we know what fields the endpoint returns. Then count by `class_name` / `class_id` to judge whether the labels are useful.

In [ ]:
print(f"total detections (post-NMS, post conf filter): {len(raw_localizations)}\n")

print("---- first 3 records (raw) ----")
for i, det in enumerate(raw_localizations[:3]):
    print(f"[{i}]", json.dumps(det, indent=2))
    print()

print("---- class_name distribution ----")
class_name_counts = Counter(
    str(d.get("class_name") or d.get("label") or d.get("name") or "(unset)")
    for d in raw_localizations
)
for name, n in class_name_counts.most_common():
    print(f"  {name!r:30s}  {n}")

print("\n---- class_id distribution ----")
class_id_counts = Counter(
    int(d.get("class_id", d.get("class", -1)) or -1)
    for d in raw_localizations
)
for cid, n in sorted(class_id_counts.items()):
    print(f"  class_id={cid:>3d}  {n}")

confs = np.array(
    [float(d.get("conf", d.get("confidence", 0.0)) or 0.0) for d in raw_localizations],
    dtype=np.float64,
)
print(
    f"\n---- confidence ---- "
    f"min={confs.min():.3f}  mean={confs.mean():.3f}  "
    f"median={np.median(confs):.3f}  max={confs.max():.3f}"
)

## 6. Visualize detections on the image

Overlay all detected bboxes on a downscaled copy of the plan (display only — bbox coordinates are scaled, not the image). Bbox color is keyed by `class_name`. Reliable detection should look:

1. Bboxes tightly enclose actual drawn symbols.
2. Few obvious false positives (boxes on text, hatching, parcel lines).
3. Few obvious false negatives (visible symbols with no box).

In [ ]:
from matplotlib.patches import Patch


def downscale(arr, max_side=2000):
    h, w = arr.shape[:2]
    s = max_side / max(h, w)
    if s >= 1.0:
        return arr.copy(), 1.0
    return cv2.resize(arr, (int(w * s), int(h * s)), interpolation=cv2.INTER_AREA), s


def render_detections(img_rgb, detections, scale=1.0, thickness=2):
    """Draw only bounding boxes (no class label, no confidence) for each
    detection. Box color is keyed by class_name so distinct classes stay
    visually separable; the color → class mapping is returned so callers
    can render an external legend.
    """
    out = img_rgb.copy()
    rng = np.random.default_rng(0)
    name_to_color: dict[str, tuple[int, int, int]] = {}

    def color_for(name: str) -> tuple[int, int, int]:
        if name not in name_to_color:
            name_to_color[name] = tuple(int(x) for x in rng.integers(60, 255, size=3))
        return name_to_color[name]

    for d in detections:
        try:
            x1 = int(float(d["x1"]) * scale); y1 = int(float(d["y1"]) * scale)
            x2 = int(float(d["x2"]) * scale); y2 = int(float(d["y2"]) * scale)
        except (KeyError, ValueError, TypeError):
            continue
        name = str(d.get("class_name") or d.get("label") or d.get("name") or "symbol")
        color = color_for(name)
        cv2.rectangle(out, (x1, y1), (x2, y2), color, thickness=thickness)
    return out, name_to_color


img_rgb = np.array(Image.open(record.path).convert("RGB"))
img_small, disp_scale = downscale(img_rgb, max_side=2000)
canvas, name_to_color = render_detections(img_small, raw_localizations, scale=disp_scale)

fig, ax = plt.subplots(figsize=(20, 14))
ax.imshow(canvas)
ax.set_title(
    f"[{SPLIT}] {record.file_name}  "
    f"({record.width}x{record.height})  --  {len(raw_localizations)} detections",
    fontsize=11,
)
ax.set_axis_off()

# Color → class legend (replaces the per-box text we used to draw).
if name_to_color:
    handles = [
        Patch(facecolor=tuple(c / 255 for c in color), edgecolor="k", label=name)
        for name, color in sorted(name_to_color.items())
    ]
    ax.legend(handles=handles, loc="upper right", fontsize=9, framealpha=0.9)

fig.tight_layout()
plt.show()

## 7. Per-class panels + confidence histogram

When there are several class labels, the all-classes overlay above can be busy. The cell below renders one panel per class — each shows only detections of that class on the plan — so it's easy to judge "do the valve boxes look like valves, do the sprinkler boxes look like sprinklers" independently.

The confidence histogram below the panels is a quick reliability check: a well-calibrated detector usually has a tall peak near high confidence with a thin left tail. A flat or bimodal histogram is a warning sign.

In [ ]:
def _name_of(d):
    return str(d.get("class_name") or d.get("label") or d.get("name") or "symbol")

names_present = sorted({_name_of(d) for d in raw_localizations})
print("classes present:", names_present)

n = len(names_present)
if n > 0:
    cols = min(3, n)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(8 * cols, 6 * rows), squeeze=False)
    for ax_idx, cls in enumerate(names_present):
        r, c = divmod(ax_idx, cols)
        subset = [d for d in raw_localizations if _name_of(d) == cls]
        panel, _ = render_detections(img_small, subset, scale=disp_scale)
        axes[r][c].imshow(panel)
        axes[r][c].set_title(f"{cls}  --  {len(subset)} detections", fontsize=10)
        axes[r][c].set_axis_off()
    # Hide unused subplots
    for k in range(n, rows * cols):
        r, c = divmod(k, cols)
        axes[r][c].set_axis_off()
    fig.tight_layout()
    plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(confs, bins=20, edgecolor="k")
ax.set_xlabel("confidence")
ax.set_ylabel("count")
ax.set_title(f"Detection confidence distribution (n={len(confs)})")
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## 8. (Optional) Run the classification endpoint on the detected crops

The classification endpoint takes the localization `crop_id`s and returns per-crop **embeddings** plus whatever other fields the server attaches (potentially a class label).

If we see usable class labels in this step (e.g., strings like `"gate-valve"`), great — we use them directly. If we only get class IDs or embeddings, we'll need either a class-id → name table from your colleague, or a small library of labelled valve / sprinkler crops to compare embeddings against.

Set `SKIP_CLASSIFY = True` to skip this step if you only care about the localization reliability check above.

In [ ]:
SKIP_CLASSIFY = False
BATCH_SIZE    = 64

cls_path = CACHE_DIR / "classifications.json"
emb_path = CACHE_DIR / "embeddings.npy"

classifications: list[dict] = []
embeddings: np.ndarray = np.empty((0, 0), dtype=np.float32)

if SKIP_CLASSIFY:
    print("skipped classification step")
elif cls_path.exists() and emb_path.exists() and not FORCE_RECOMPUTE:
    classifications = json.loads(cls_path.read_text())
    embeddings = np.load(emb_path)
    print(f"loaded {len(classifications)} classifications and embeddings of shape {embeddings.shape}")
else:
    # Collect crop_ids from raw localizations.
    crop_ids: list[str] = []
    for d in raw_localizations:
        cid = d.get("id") or d.get("ID") or d.get("crop_id")
        if cid is not None:
            crop_ids.append(str(cid))
    print(f"classifying {len(crop_ids)} crops in batches of {BATCH_SIZE}...")

    emb_rows: list[list[float]] = []
    for start in range(0, len(crop_ids), BATCH_SIZE):
        chunk = crop_ids[start:start + BATCH_SIZE]
        result = client.classify(chunk) or []
        if len(result) != len(chunk):
            raise RuntimeError(
                f"classify returned {len(result)} rows for {len(chunk)} crop_ids"
            )
        for crop_id, rec in zip(chunk, result):
            if not isinstance(rec, dict):
                rec = {"raw": rec}
            emb = rec.get("embedding") or rec.get("embeddings") or rec.get("EMBEDDINGS")
            saved = {k: v for k, v in rec.items() if k not in ("embedding", "embeddings", "EMBEDDINGS")}
            if emb is not None:
                saved["embedding_dim"] = len(emb)
                emb_rows.append([float(v) for v in emb])
            classifications.append({"crop_id": crop_id, "classification": saved})
    embeddings = (
        np.asarray(emb_rows, dtype=np.float32)
        if emb_rows else np.empty((0, 0), dtype=np.float32)
    )
    cls_path.write_text(json.dumps(classifications, indent=2))
    if embeddings.size:
        np.save(emb_path, embeddings)
    print(f"saved {len(classifications)} classifications, embeddings shape {embeddings.shape}")

## 9. Peek at classification output

What keys does each classification record carry — class labels, scores, embeddings only?

In [ ]:
if not classifications:
    print("no classifications available")
else:
    sample = classifications[0]
    inner = sample.get("classification", {})
    print("outer keys:", list(sample.keys()))
    print("inner keys:", list(inner.keys()))
    print("\nfirst record (without embedding):")
    print(json.dumps(sample, indent=2)[:1500])

    if embeddings.size:
        norms = np.linalg.norm(embeddings, axis=1)
        print(
            f"\nembeddings: shape={embeddings.shape}, dtype={embeddings.dtype}  "
            f"per-row norm  min={norms.min():.3f}  mean={norms.mean():.3f}  max={norms.max():.3f}"
        )

## 10. Cleanup

Remove the temporary GCS blob the client uploaded during this session. Safe to skip if you want to inspect the blob in the bucket — just remember to call this later (or rely on the `finally` block in the CLI version of the script).

In [ ]:
client.cleanup_uploads()
print("cleaned up any temporary GCS uploads")